# Create Worker Prior Experience Counts
This notebook processes the raw position history dataset and calculates worker-level prior work experience in each industry and at each level of seniority, calculated strictly prior to their first founding event.

## Objectives & Methodology
1. **Pre-Entrepreneurship Threshold**: We calculate the start date of the first founding event (`first_founding_date`) for each user. Only experience in positions starting *before* this date is considered.
2. **Experience Capping**: If a worker holds a position that starts before their first founding event but ends after it, its duration is capped at `first_founding_date` (duration = `first_founding_date` - `startdate`).
3. **Pivoted Output (One Row Per Worker)**: To make it easy to match this experience history with the education and events datasets, we pivot the experience rows into two final wide-format datasets:
   * **Industry Experience**: Unique column for each industry classification (`rics_k400`), e.g. `exp_ind_Software`.
   * **Seniority Experience**: Columns `exp_sen_1` through `exp_sen_7` representing years spent at each seniority level.
4. **Memory-Efficient Sharding**: Uses the 20 partitioned shards generated by the sharding script to process the massive position list.


In [1]:
import pandas as pd
import numpy as np
import os
import time
import csv
import re
import shutil

base_dir = ".."
positions_path = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/Founder_Full_Position_List_[US-2000-2023].csv")
mapping_path = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/Founder_Position_List_[US-2000-2023].csv")
shard_dir = os.path.join(base_dir, "D - Data/D1 - Extracted Datasets/temp_shards")
out_dir = os.path.join(base_dir, "D - Data/D2 - Datasets for Matching")

os.makedirs(out_dir, exist_ok=True)

# Load mapping set of founding positions
print("Loading founding position IDs...")
df_map = pd.read_csv(mapping_path, usecols=['position_id'], low_memory=False)
founding_positions = set(df_map['position_id'].values)

# Helper to parse dates
def parse_date(date_val, year_val):
    if pd.notnull(date_val):
        date_str = str(date_val).strip()
        if date_str and date_str.lower() not in ['empty', 'nan', 'none']:
            try:
                return pd.to_datetime(date_str)
            except Exception:
                pass
    if pd.notnull(year_val):
        try:
            yr = int(float(year_val))
            if 1950 <= yr <= 2026:
                return pd.Timestamp(year=yr, month=1, day=1)
        except Exception:
            pass
    return None

def process_shard_for_experience(shard_path, founding_positions):
    df_shard = pd.read_csv(shard_path, low_memory=False)
    
    # Parse dates
    df_shard['parsed_start'] = df_shard.apply(lambda r: parse_date(r['startdate'], r['startyear']), axis=1)
    df_shard['parsed_end'] = df_shard.apply(lambda r: parse_date(r['enddate'], r['endyear']), axis=1)
    
    # Sort
    df_shard = df_shard.sort_values(by=['user_id', 'parsed_start', 'parsed_end']).copy()
    
    # Found_Pos indicator
    df_shard['is_founding'] = df_shard['position_id'].apply(lambda pid: 1 if pid in founding_positions else 0)
    
    industry_exp_rows = []
    seniority_exp_rows = []
    
    grouped = df_shard.groupby('user_id')
    for user_id, group in grouped:
        positions = group.to_dict('records')
        
        # Find first founding date
        first_founding_date = None
        for pos in positions:
            if pos['is_founding'] == 1:
                if pos['parsed_start'] is not None:
                    if first_founding_date is None or pos['parsed_start'] < first_founding_date:
                        first_founding_date = pos['parsed_start']
                        
        if first_founding_date is None:
            continue
            
        # Accumulate prior experience
        industry_durations = {}
        seniority_durations = {}
        
        for pos in positions:
            # Only consider positions starting before the first founding date
            if pos['is_founding'] == 0 and pos['parsed_start'] is not None and pos['parsed_start'] < first_founding_date:
                pos_end = pos['parsed_end'] if pd.notnull(pos['parsed_end']) else first_founding_date
                if pos_end > first_founding_date:
                    pos_end = first_founding_date
                    
                pos_duration = (pos_end - pos['parsed_start']).days / 365.25
                if pos_duration < 0:
                    pos_duration = 0.0
                    
                # Accumulate industry (rics_k400)
                ind = pos['rics_k400']
                if pd.notnull(ind) and str(ind).strip() != '' and str(ind).lower() != 'empty':
                    ind_str = str(ind).strip()
                    industry_durations[ind_str] = industry_durations.get(ind_str, 0.0) + pos_duration
                    
                # Accumulate seniority
                sen = pos['seniority']
                try:
                    sen_int = int(float(sen))
                    seniority_durations[sen_int] = seniority_durations.get(sen_int, 0.0) + pos_duration
                except:
                    pass
                    
        for ind, exp in industry_durations.items():
            industry_exp_rows.append({'user_id': user_id, 'industry': ind, 'experience_years': exp})
        for sen, exp in seniority_durations.items():
            seniority_exp_rows.append({'user_id': user_id, 'seniority_level': sen, 'experience_years': exp})
            
    return industry_exp_rows, seniority_exp_rows

# Process all shards
all_industry_exp = []
all_seniority_exp = []
start_time = time.time()

if not os.path.exists(shard_dir) or len(os.listdir(shard_dir)) == 0:
    print("Shard directory empty or not found. Please run the sharding script first.")
else:
    shard_files = sorted([f for f in os.listdir(shard_dir) if f.startswith("shard_")])
    for idx, f in enumerate(shard_files):
        shard_path = os.path.join(shard_dir, f)
        print(f"Processing shard {idx+1}/{len(shard_files)} ({f})...")
        ind_exp, sen_exp = process_shard_for_experience(shard_path, founding_positions)
        all_industry_exp.extend(ind_exp)
        all_seniority_exp.extend(sen_exp)
        
    print(f"Processed all shards in {time.time() - start_time:.2f} seconds.")

# Process and save pivoted experience datasets
if all_industry_exp:
    print("Pivoting industry experience dataset...")
    df_ind_raw = pd.DataFrame(all_industry_exp)
    
    # Pivot
    df_ind_piv = df_ind_raw.pivot(index='user_id', columns='industry', values='experience_years').fillna(0.0)
    
    # Rename columns to standard safe names
    # e.g. "Software and Tech Services" -> "exp_ind_Software_and_Tech_Services"
    new_cols = []
    for c in df_ind_piv.columns:
        c_clean = re.sub(r'[^\w]', '_', str(c)).strip('_')
        # collapse multiple underscores
        c_clean = re.sub(r'_+', '_', c_clean)
        new_cols.append(f"exp_ind_{c_clean}")
    df_ind_piv.columns = new_cols
    df_ind_piv = df_ind_piv.reset_index()
    
    ind_out_path = os.path.join(out_dir, "Founder_Prior_Experience_Industry.csv")
    df_ind_piv.to_csv(ind_out_path, index=False)
    print(f"Saved industry experience to: {ind_out_path}")
    print(f"Industry exp shape: {df_ind_piv.shape}")

if all_seniority_exp:
    print("Pivoting seniority experience dataset...")
    df_sen_raw = pd.DataFrame(all_seniority_exp)
    
    df_sen_piv = df_sen_raw.pivot(index='user_id', columns='seniority_level', values='experience_years').fillna(0.0)
    df_sen_piv.columns = [f"exp_sen_{int(float(c))}" for c in df_sen_piv.columns]
    df_sen_piv = df_sen_piv.reset_index()
    
    sen_out_path = os.path.join(out_dir, "Founder_Prior_Experience_Seniority.csv")
    df_sen_piv.to_csv(sen_out_path, index=False)
    print(f"Saved seniority experience to: {sen_out_path}")
    print(f"Seniority exp shape: {df_sen_piv.shape}")

# Optional: clean up temporary shards to free space
if os.path.exists(shard_dir):
    print("Cleaning up temporary shard files...")
    shutil.rmtree(shard_dir)
    print("Cleanup complete.")



Loading founding position IDs...
Processing shard 1/20 (shard_0.csv)...


Processing shard 2/20 (shard_1.csv)...


Processing shard 3/20 (shard_10.csv)...


Processing shard 4/20 (shard_11.csv)...


Processing shard 5/20 (shard_12.csv)...


...

Processing shard 19/20 (shard_8.csv)...


Processing shard 20/20 (shard_9.csv)...


Processed all shards in 2057.04 seconds.
Pivoting industry experience dataset...


Saved industry experience to: ../D - Data/D2 - Datasets for Matching/Founder_Prior_Experience_Industry.csv
Industry exp shape: (803184, 401)
Pivoting seniority experience dataset...


Saved seniority experience to: ../D - Data/D2 - Datasets for Matching/Founder_Prior_Experience_Seniority.csv
Seniority exp shape: (876800, 8)
Cleaning up temporary shard files...
Cleanup complete.
